<a href="https://colab.research.google.com/github/u11210013/HF_Seismic_Wave_Recognition/blob/main/seismic_wave_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install obspy
!pip install obspy-taup
!pip install datasets
!pip install huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 61.6 MB/s eta 0:00:00


ERROR: Could not find a version that satisfies the requirement obspy-taup (from versions: none)
ERROR: No matching distribution found for obspy-taup


抓資料

In [ ]:
import time
import numpy as np
from obspy.clients.fdsn import Client
from obspy import UTCDateTime, Stream
from obspy.taup import TauPyModel
from obspy.geodetics import locations2degrees
from obspy.signal.trigger import classic_sta_lta, trigger_onset
import matplotlib.pyplot as plt
from IPython.display import clear_output, display
from datasets import Dataset, DatasetDict # 引入 Hugging Face Datasets

# ==========================================
# 1. 建立 FDSN 客戶端與獲取地震目錄
# ==========================================
client = Client("EARTHSCOPE")
t1 = UTCDateTime("2025-10-10T20:20:00")
t2 = UTCDateTime("2025-10-10T21:20:00")
catalog = client.get_events(starttime=t1, endtime=t2, minmagnitude=7.0)

event = catalog[0]
origin = event.origins[0]

event_time = origin.time
event_lat = origin.latitude
event_lon = origin.longitude
event_depth_km = origin.depth / 1000.0

# ==========================================
# 2. 獲取測站清單並由近到遠排序
# ==========================================
print("正在向 EARTHSCOPE 請求測站清單...")
try:
    inventory = client.get_stations(latitude=event_lat, longitude=event_lon, maxradius=30,
                                    channel="BHZ", starttime=event_time, endtime=event_time + 3600,
                                    level="station")
except Exception as e:
    print(f"獲取測站清單失敗: {e}")
    exit()

station_list = []
for network in inventory:
    for sta in network:
        dist = locations2degrees(event_lat, event_lon, sta.latitude, sta.longitude)
        station_list.append({
            'network': network.code,
            'station': sta.code,
            'lat': sta.latitude,
            'lon': sta.longitude,
            'distance': dist
        })

station_list = sorted(station_list, key=lambda x: x['distance'])

# ==========================================
# 3. 第一階段：自動下載 (抓取 10 個測站)
# ==========================================
model = TauPyModel(model="iasp91")
st_all = Stream()
success_count = 0

sta_sec, lta_sec = 1.0, 5.0
thr_on, thr_off = 4.5, 2.0

for sta in station_list:
    if success_count >= 20: # 改成挑選 10 個
        break

    net_code, sta_code, dist = sta['network'], sta['station'], sta['distance']
    arrivals = model.get_travel_times(source_depth_in_km=event_depth_km,
                                      distance_in_degree=dist,
                                      phase_list=["P", "Pn", "Pg", "p"])
    if not arrivals: continue

    p_arrival_time = event_time + arrivals[0].time
    start_window, end_window = p_arrival_time - 20, p_arrival_time + 100

    try:
        st = client.get_waveforms(network=net_code, station=sta_code, location="*",
                                  channel="BHZ", starttime=start_window, endtime=end_window)
        if len(st) == 0: continue

        unique_ids = list(set([tr.id for tr in st]))
        valid_trace_found = False

        for tr_id in unique_ids:
            st_id = st.select(id=tr_id)
            if len(st_id) > 1: continue

            tr = st_id[0]
            if abs(tr.stats.sampling_rate - 20.0) > 0.001: continue
            if (tr.stats.endtime - tr.stats.starttime) < 119.0: continue

            tr_test = tr.copy()
            tr_test.detrend("demean")
            df = tr_test.stats.sampling_rate
            cft = classic_sta_lta(tr_test.data, int(sta_sec * df), int(lta_sec * df))
            triggers = trigger_onset(cft, thr_on, thr_off)

            if len(triggers) == 0: continue

            tr.stats.p_arrival = p_arrival_time
            st_all.append(tr)
            valid_trace_found = True
            break

        if valid_trace_found:
            success_count += 1
            print(f"[自動篩選] {success_count}/20 成功抓取 {net_code}.{sta_code}")

    except Exception:
        pass

time.sleep(1.5)

# ==========================================
# 4. 第二階段：人工審核並建立資料集特徵
# ==========================================
# 準備給 Hugging Face 的字典格式
hf_data_dict = {
    "text": [],   # 存放波形數值陣列
    "label": []   # 存放標籤 (0 或 1)
}

for tr in st_all:
    clear_output(wait=True)
    print(f"🔍 正在檢視測站: {tr.id}")
    fig = tr.plot(handle=True, title=f"Reviewing {tr.id}")
    display(fig)
    time.sleep(0.5)

    choice = input(f"此波形是否為有效地震訊號? (輸入 y 給標籤 1 / 其他鍵給標籤 0): ")
    plt.close(fig)

    # --- 不管是 0 還是 1，都必須裁切出相同長度 (100秒) 以供模型訓練 ---
    p_time = tr.stats.p_arrival
    cut_start = p_time - 10
    cut_end = p_time + 90
    tr_cut = tr.slice(starttime=cut_start, endtime=cut_end)

    # 將波形的振幅數據 (Numpy array) 轉為 Python List
    # 如果未來需要標準化，可以在這裡加入正規化處理 (例如除以最大值)
    amplitude_data = tr_cut.data.tolist()

    # 決定標籤
    label = 1 if choice.strip().lower() == 'y' else 0

    hf_data_dict["text"].append(amplitude_data)
    hf_data_dict["label"].append(label)

# ==========================================
# 5. 轉換為 Hugging Face DatasetDict
# ==========================================
clear_output(wait=True)

# 步驟 A: 將 Dictionary 轉成 Hugging Face 的 Dataset 物件
full_dataset = Dataset.from_dict(hf_data_dict)

# 步驟 B: 將這 10 筆資料切割成 train, validation, test
# (因為只有 10 筆，這裡示範切成 8 筆 train，1 筆 validation，1 筆 test)
train_test_split = full_dataset.train_test_split(test_size=0.2, seed=42)
val_test_split = train_test_split['test'].train_test_split(test_size=0.5, seed=42)

ds = DatasetDict({
    'train': train_test_split['train'],
    'validation': val_test_split['train'],
    'test': val_test_split['test']
})

print("=== 🎉 恭喜！Hugging Face DatasetDict 建立完成 ===")
print(ds)

# 您可以試著印出第一筆訓練資料來檢查格式：
print("\n[預覽] 訓練集第一筆資料的標籤:", ds['train'][0]['label'])
print("[預覽] 訓練集第一筆資料的波形資料點數量:", len(ds['train'][0]['text']))

In [ ]:
import time
import numpy as np
from obspy.clients.fdsn import Client
from obspy import UTCDateTime, Stream
from obspy.taup import TauPyModel
from obspy.geodetics import locations2degrees
import matplotlib.pyplot as plt
from IPython.display import clear_output, display
from datasets import Dataset, DatasetDict

# ==========================================
# 1. 初始化與獲取地震目錄 (EarthScope)
# ==========================================
client = Client("EARTHSCOPE")
# 以 2024 能登地震為例，您可以更換時間與座標
t1 = UTCDateTime("2025-10-10T20:20:00")
t2 = UTCDateTime("2025-10-10T21:20:00")
catalog = client.get_events(starttime=t1, endtime=t2, minmagnitude=7.0)

event = catalog[0]
origin = event.origins[0]
event_time, event_lat, event_lon = origin.time, origin.latitude, origin.longitude
event_depth_km = origin.depth / 1000.0

print(f"=== 地震事件資訊 ===")
print(f"發震時間: {event_time} | 座標: {event_lat}, {event_lon} | 深度: {event_depth_km} km\n")

# ==========================================
# 2. 獲取震央距 30 度內的所有測站
# ==========================================
print("正在檢索 30 度內所有 BHZ 測站清單...")
try:
    inventory = client.get_stations(latitude=event_lat, longitude=event_lon, maxradius=30,
                                    channel="BHZ", starttime=event_time, endtime=event_time + 3600,
                                    level="station")
except Exception as e:
    print(f"獲取測站清單失敗: {e}"); exit()

station_list = []
for network in inventory:
    for sta in network:
        dist = locations2degrees(event_lat, event_lon, sta.latitude, sta.longitude)
        station_list.append({
            'network': network.code, 'station': sta.code, 'distance': dist
        })

# 由近到遠排序
station_list = sorted(station_list, key=lambda x: x['distance'])
print(f"找到共 {len(station_list)} 個候選測站。開始進行品質篩選 (20Hz, 無中斷)...")

# ==========================================
# 3. 第一階段：自動品質篩選 (20Hz + 無中斷)
# ==========================================
model = TauPyModel(model="iasp91")
st_to_review = Stream()

for i, sta in enumerate(station_list):
    net_code, sta_code, dist = sta['network'], sta['station'], sta['distance']

    # 計算 P 波走時
    arrivals = model.get_travel_times(source_depth_in_km=event_depth_km,
                                      distance_in_degree=dist,
                                      phase_list=["P", "Pn", "Pg", "p"])
    if not arrivals: continue
    p_arrival = event_time + arrivals[0].time

    # 預抓寬一點 (-20 ~ +100) 以便之後精準裁切
    try:
        st = client.get_waveforms(network=net_code, station=sta_code, location="*",
                                  channel="BHZ", starttime=p_arrival - 20, endtime=p_arrival + 100)

        # 遍歷下載到的 Trace (處理同測站多儀器位置情況)
        for tr in st:
            # A. 檢查取樣率是否為 20Hz
            if abs(tr.stats.sampling_rate - 20.0) > 0.001: continue

            # B. 檢查是否有分段 (在該時間範圍內此 ID 只能有一條 Trace)
            if len(st.select(id=tr.id)) > 1: continue

            # C. 檢查長度是否足夠 (120秒 = 2400 點)
            if tr.stats.npts < 2350: continue

            # 通過初步篩選，暫存 P 波時間並加入待審核清單
            tr.stats.p_arrival = p_arrival
            st_to_review.append(tr)

    except:
        pass

    if i % 10 == 0:
        print(f"進度: {i}/{len(station_list)} 測站已掃描...", end="\r")

print(f"\n\n篩選完成！共 {len(st_to_review)} 條波形符合 20Hz 且無中斷條件。")
print("進入人工審核階段...")
time.sleep(2)

# ==========================================
# 4. 第二階段：人工審核並製作 Dataset
# ==========================================
hf_data_dict = {"text": [], "label": []}

for tr in st_to_review:
    clear_output(wait=True)
    print(f"🔍 審核中 ({len(hf_data_dict['label'])+1}/{len(st_to_review)}): {tr.id}")
    print("👉 y: 地震訊號 (Label 1) | n: 雜訊/捨棄 (Label 0)")

    # 繪圖檢視 (顯示完整預抓範圍)
    fig = tr.plot(handle=True, title=f"{tr.id} - Verify P-wave Window")
    display(fig)
    time.sleep(0.5)

    choice = input("保留此波形嗎? (y/n): ").strip().lower()
    plt.close(fig)

    # 統一進行裁切 (P-10s ~ P+90s)
    p_t = tr.stats.p_arrival
    tr_cut = tr.slice(starttime=p_t - 10, endtime=p_t + 90)

    # 確保資料點數量一致 (20Hz * 100s = 2000點)
    # 若因邊界微差導致 npts 不同，強迫截斷或補零
    data_list = tr_cut.data[:2000].tolist()
    if len(data_list) < 2000:
        # 若長度不足補最後一個值的數值 (Padding)
        data_list.extend([data_list[-1]] * (2000 - len(data_list)))

    # 存入字典
    hf_data_dict["text"].append(data_list)
    hf_data_dict["label"].append(1 if choice == 'y' else 0)

# ==========================================
# 5. 轉換為 Hugging Face DatasetDict
# ==========================================
clear_output(wait=True)
full_ds = Dataset.from_dict(hf_data_dict)

# 根據總筆數自動計算切分比例 (若筆數太少會報錯，這裡加個簡單判斷)
num_samples = len(full_ds)
if num_samples >= 3:
    train_test = full_ds.train_test_split(test_size=0.2, seed=42)
    val_test = train_test['test'].train_test_split(test_size=0.5, seed=42)
    ds = DatasetDict({
        'train': train_test['train'],
        'validation': val_test['train'],
        'test': val_test['test']
    })
else:
    ds = DatasetDict({'train': full_ds})
    print("警告: 樣本數過少，僅建立 train dataset。")

print("=== 🎉 DatasetDict 建立完成 ===")
print(ds)

# 顯示最後結果預覽
if num_samples > 0:
    print(f"\n單一波形特徵長度: {len(ds['train'][0]['text'])} 點")
    print(f"標籤分佈: 1(Yes)={hf_data_dict['label'].count(1)}, 0(No)={hf_data_dict['label'].count(0)}")

Huggingface

In [ ]:
from datasets import load_dataset

# 抓取你之前上傳的所有成果
old_ds = load_dataset("u11210013/earthquake_dataset_project")

In [ ]:
from datasets import concatenate_datasets, DatasetDict

# 分別合併 train, validation, test
merged_ds = DatasetDict({
    'train': concatenate_datasets([old_ds['train'], ds['train']]),
    'validation': concatenate_datasets([old_ds['validation'], ds['validation']]),
    'test': concatenate_datasets([old_ds['test'], ds['test']])
})

In [ ]:
from google.colab import userdata
from huggingface_hub import login

# 登入 (使用你 Secrets 裡的 Token)
login(token=userdata.get('HF_token'))

# 推送更新
merged_ds.push_to_hub("u11210013/earthquake_dataset_project")

print(f"✅ 更新成功！目前總樣本數：{len(merged_ds['train'])}")

上傳github

In [ ]:
from google.colab import userdata
token = userdata.get('Github_token')

# 然後組合 URL
repo_url = f"https://{token}@github.com/u11210013/HF_Seismic_Wave_Recognition.git"
!git clone {repo_url}

Cloning into 'HF_Seismic_Wave_Recognition'...


In [10]:
# 1. 初始化 (如果 clone 下來已經有 .git 資料夾則免，但跑一下沒壞處)
!git init

# 2. 將檔案加入暫存區
!git add .

# 3. 提交變更
!git commit -m "initial commit"

# 5. 推送回 GitHub
!git push -u origin main

Reinitialized existing Git repository in /content/.git/
hint: You've added another git repository inside your current repository.
hint: Clones of the outer repository will not contain the contents of
hint: the embedded repository and will not know how to obtain it.
hint: If you meant to add a submodule, use:
hint: 
hint: 	git submodule add <url> HF_Seismic_Wave_Recognition
hint: 
hint: If you added this path by mistake, you can remove it from the
hint: index with:
hint: 
hint: 	git rm --cached HF_Seismic_Wave_Recognition
hint: 
hint: See "git help submodule" for more information.
[main e4f55e0] initial commit
 1 file changed, 1 insertion(+)
 create mode 160000 HF_Seismic_Wave_Recognition
To https://github.com/u11210013/HF_Seismic_Wave_Recognition
 ! [rejected]        main -> main (fetch first)
error: failed to push some refs to 'https://github.com/u11210013/HF_Seismic_Wave_Recognition'
hint: Updates were rejected because the remote contains work that you do
hint: not have locally. This